# Langchain Tutorial 

https://docs.langchain.com/oss/python/langchain/rag#loading-documents


# To do

- Embedded all the txt files, use chroma db
- Create a RAG architecture (fix k)
- Connect it to my telegram
- Add a feature to store any new doc in the db

In [1]:
import os
from os import listdir
from os.path import isfile, join
from dotenv import load_dotenv

In [2]:
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

c:\Users\charles\OneDrive\Projet Perso\JJBTelegram\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
model = init_chat_model("claude-sonnet-4-6")

In [5]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name="microsoft/harrier-oss-v1-270m")

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 1202.01it/s]
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [8]:
vector_store = Chroma(
    collection_name="jjb_notes",
    embedding_function=embeddings,
    persist_directory="data/chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [4]:
vector_store = Chroma(
    collection_name="jjb_notes",
    embedding_function=embeddings,
    persist_directory="data/test_db",  # Where to save data locally, remove if not necessary
)

In [10]:
nb_docs = vector_store._collection.count()
type(nb_docs)

int

# Embedding documents

## Load txt files

In [5]:
f1 = open('data/txt/2025_10_02_Russian_tie_system.txt', 'r', encoding='utf-8')
f2 = open('data/txt/2026_03_24_Mount_attacks.txt', 'r', encoding='utf-8')

In [6]:
content1 = f1.read()
content2 = f2.read()

In [7]:
print(f"content1 = {content1}")

content1 = Title: Russian tie system
Date: 02/10/2025
Type: jjb
Created: 2026-03-21T14:59:56.483730

────────────────────────────────────────

[ Page 1 ]
SÉANCE #2                                   21/10/25

Depuis le russian tie, mettre de la pression
sur son épaule : "classic foot sweep" : sa jambe
la plus proche de moi, je passe mon pied derrière
son pied & plus proche de moi. J'oriente mon
pied comme si je faisais un plat du pied.
Je pousse son pied dans le sens de ses orteils
et j'essaye de lever sa jambe le + haut
possible. Continuer à mettre de la pression
sur son épaule.

(1) Il Carre en arrière
(2) Il sort sa jambe. En sortant sa jambe,
il risque de baisser sa tête → front head
lock.

o Système Russian tie :

(1) Attaque sur son bras :
- Drag down : mettre la pression sur son élan
état et le plaquer au sol.

[ Page 2 ]
- suite par paquet dans l'intérieur de ma jambe
le ⊕ posée de lui - Reculer à double pied en faisant pression
sur sa garde

(2) Attaque de jambe :
- Si la perso

In [8]:
print(f"{content2}")

Title: Mount attacks
Date: 24/03/2026
Type: jjb
Created: 2026-03-28T16:28:29.137492

────────────────────────────────────────

[ Page 1 ]
SÉANCE 114 - 24/03/20

Attaques depuis la guard :

(1) Il défend en tendant ses bras sur mes fees.
Je ne laisse tomber en avant pour que ses
mains passent au dessus de sa tête...
Je choisis le côté que je vais attaquer. Supposons
que j'attaque son bras logside droit.
Je remonte ma jambe gauche haut pour bloquer
son bras et qu'il ne puisse plus le redescendre.
Ma main droite vient sous son épaule gauche
et je prends la S- mount.

Je tiens mettre tout mon poids sur sa
tête

Ma main gauche ensuite son ses bras que je
vais attaquer. Pour pouvoir passer ma jambe
gauche au dessus de sa tête, je viens poser
ma main droite au sol au niveau de sa
hanche. Je déplie ma jambe gauche, je
passe au dessus de sa tête et je mets mon talon
contre sa tête. Finaliser l'armbar.
Travailler le positiof de mes jambes et
mon cops par qu'il ne bouge pas.

[ Page 2 ]
(2) Améri

In [9]:
list_doc_str = [content1, content2]

doc_embedded = embeddings.embed_documents(list_doc_str)

In [ ]:
document_1 = Document(id="1", page_content=content1, metadata={"title": "Russian tie system", 'date': '02/10/2025'})
document_2 = Document(id="2", page_content=content2, metadata={"title": "Mount attacks", 'date': '24/03/2026' })

documents = [document_1, document_2]
vector_store.add_documents(documents=documents)

['1', '2']

## Add a doc

In [ ]:
f3 = open('data/txt/2026_04_19_Collar_tie.txt', 'r', encoding='utf-8')
content3 = f3.read()

In [12]:
document_3 = Document(id="3", 
                page_content=content3, 
                metadata={"title": "Collar tie", "date" : "06/11/2025"})

In [13]:
vector_store.add_documents(documents=[document_3])

['3']

In [15]:
results = vector_store.similarity_search_with_score(query="Quelles sont les attaques depuis la mount?", k=1)
for doc, score in results:
    print(f"* [SIM={score:3f}]\n{doc.page_content}\n[{doc.metadata}]")

* [SIM=0.697313]
Title: Mount attacks
Date: 24/03/2026
Type: jjb
Created: 2026-03-28T16:28:29.137492

────────────────────────────────────────

[ Page 1 ]
SÉANCE 114 - 24/03/20

Attaques depuis la guard :

(1) Il défend en tendant ses bras sur mes fees.
Je ne laisse tomber en avant pour que ses
mains passent au dessus de sa tête...
Je choisis le côté que je vais attaquer. Supposons
que j'attaque son bras logside droit.
Je remonte ma jambe gauche haut pour bloquer
son bras et qu'il ne puisse plus le redescendre.
Ma main droite vient sous son épaule gauche
et je prends la S- mount.

Je tiens mettre tout mon poids sur sa
tête

Ma main gauche ensuite son ses bras que je
vais attaquer. Pour pouvoir passer ma jambe
gauche au dessus de sa tête, je viens poser
ma main droite au sol au niveau de sa
hanche. Je déplie ma jambe gauche, je
passe au dessus de sa tête et je mets mon talon
contre sa tête. Finaliser l'armbar.
Travailler le positiof de mes jambes et
mon cops par qu'il ne bouge pas.

[ P

## Test query

In [9]:
query = "quelles sont les meilleures attaques depuis la mount ?"
#query_embedding = embeddings.embed_query(query)

In [12]:
results = vector_store.similarity_search_with_score(query=query, k=3)
for doc, score in results[::-1]:
    print(f"* [SIM={score:3f}]\n{doc.page_content[:40]}\n[{doc.metadata}]")

* [SIM=0.869428]
Title: Mount get control submit
Date: 09
[{'title': 'Mount get control submit', 'date': '09/02/2026'}]
* [SIM=0.846592]
Title: Lutte & defense mount & dog fight
[{'date': '10/10/2025', 'title': 'Lutte & defense mount & dog fight'}]
* [SIM=0.708991]
Title: Mount attacks
Date: 24/03/2026
Ty
[{'date': '24/03/2026', 'title': 'Mount attacks'}]


# Create db

In [5]:
path_txt = "data/txt"
all_txt = [f for f in listdir(path_txt) if isfile(join(path_txt, f))]


In [6]:
activity = all_txt[30][11:-4]
activity.replace('_', ' ')

'Defense collar tie & get out of closed guard'

In [7]:
documents = []
for cpt, note_path in enumerate(all_txt):

    note_txt = open(path_txt+'/'+note_path, 'r', encoding='utf-8')
    content = note_txt.read()
    date = note_path[:10]
    date = date[-2:] + '/' + date[-5:-3] + '/' + date[:4]
    theme = note_path[11:-4].replace('_', ' ')
    doc = Document(id=str(cpt+1), page_content=content, metadata={"title": theme, 'date': date})

    documents.append(doc)

In [8]:
vector_store.add_documents(documents=documents)

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47']

In [9]:
len(f"Here is your document:\n\n\nSend rank number if you want to see a particular document.\nSend *QUERY* if you want to ask something else.\nSend *GO* otherwise")
            

151